# Séance 2 — Pandas : structures et exploration

**Durée pratique : 3 h 30** &nbsp;·&nbsp; Ateliers 2.1 à 2.3

## Ce que vous saurez faire à la fin

- charger une même source depuis trois formats et réconcilier les types obtenus ;
- sélectionner et filtrer sans déclencher de `SettingWithCopyWarning` ;
- produire automatiquement un rapport d'exploration sur un jeu de données inconnu.

Le jeu de travail est `ventes_brutes.csv` : environ 17 600 lignes de commandes,
volontairement dégradées. Vous allez le retrouver à chaque séance jusqu'à la fin du module.

> **Convention de nommage du module.** Le code est écrit en anglais et suit la PEP 8 :
> fonctions et variables en `snake_case`, constantes en `MAJUSCULES`. Les **noms de colonnes**
> restent en français parce qu'ils viennent de la source : renommer les colonnes d'un fichier
> d'entrée est une transformation comme une autre, elle se décide et se documente, elle ne se
> fait pas par réflexe. Vous rencontrerez cette situation partout en entreprise.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

print('Données disponibles :')
for path in sorted(RAW_DIR.glob('*')):
    print(' ', path.name)


# Parquet conserve les types (dates, entiers, catégories) là où le CSV les perd :
# c'est le format à privilégier entre deux étapes d'un pipeline. Repli automatique
# sur le CSV si pyarrow n'est pas installé.
def save_dataset(df, name):
    try:
        path = PROCESSED_DIR / f'{name}.parquet'
        df.to_parquet(path, index=False)
    except ImportError:
        path = PROCESSED_DIR / f'{name}.csv'
        df.to_csv(path, index=False)
        print('(pyarrow absent : repli sur le CSV)')
    print('écrit :', path.name, df.shape)
    return path


def load_dataset(name):
    parquet_path = PROCESSED_DIR / f'{name}.parquet'
    csv_path = PROCESSED_DIR / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'{name} introuvable : exécutez le notebook précédent')


def dataset_exists(name):
    return ((PROCESSED_DIR / f'{name}.parquet').exists()
            or (PROCESSED_DIR / f'{name}.csv').exists())

Données disponibles :
  .gitkeep
  capteurs.csv
  clients.csv
  magasins.csv
  produits.csv
  ventes_brutes.csv
  ventes_extrait.csv
  ventes_extrait.json
  ventes_extrait.xlsx


---
## Atelier 2.1 — Lire une source, vraiment (50 min)

### Partie guidée : la lecture naïve et ce qu'elle cache

In [2]:
sales = pd.read_csv(RAW_DIR / 'ventes_brutes.csv')

print('Dimensions :', sales.shape)
sales.head()

Dimensions : (17864, 9)


,id_commande,date_commande,id_client,id_magasin,id_produit,quantite,prix_unitaire,statut,canal
0,CMD000001,2023-02-04,C0573,M06,P0032,11,"829,94 EUR",annule,WEB
1,CMD000002,27/02/2023,C2949,M03,P0017,2,488.34,livre,WEB
2,CMD000003,2021-04-02,C2965,M03,P0008,2,"738,30 EUR",annule,telephone
3,CMD000004,18 Apr 2023,C1252,M07,P0032,1,951.28,en_cours,telephone
4,CMD000005,2021/02/10 15:42,C2685,M02,P0016,3,248.84,livre,boutique


In [3]:
sales.dtypes

id_commande        str
date_commande      str
id_client          str
id_magasin         str
id_produit         str
quantite         int64
prix_unitaire      str
statut             str
canal              str
dtype: object

Regardez `prix_unitaire`. Le type est `object`, autrement dit du texte, alors qu'il s'agit
d'un montant. Cherchons pourquoi.

In [4]:
# Isoler les valeurs qui ne se convertissent pas en nombre
as_number = pd.to_numeric(sales['prix_unitaire'], errors='coerce')
unparsable = sales.loc[as_number.isna(), 'prix_unitaire']

print('Valeurs non convertibles :', len(unparsable))
print(unparsable.head(8).tolist())

Valeurs non convertibles : 4518
['829,94 EUR', '738,30 EUR', '107,10 EUR', '423,56 EUR', '760,17 EUR', '684,22 EUR', '946,38 EUR', '731,56 EUR']


Une partie des prix a été saisie avec une virgule décimale et un symbole monétaire.
Une lecture qui ignore ce détail produit une colonne texte, et tout calcul en aval échoue
silencieusement ou renvoie une erreur bien plus loin dans le pipeline.

**C'est la règle à retenir : ne jamais faire confiance à l'inférence de types.** Vérifiez
systématiquement `dtypes` après chaque lecture.

In [5]:
def parse_price(series):
    """Convertit une colonne de prix mixte (nombre ou texte '123,45 EUR') en float."""
    text = series.astype(str).str.replace(' EUR', '', regex=False)
    text = text.str.replace(',', '.', regex=False).str.strip()
    return pd.to_numeric(text, errors='coerce')


unit_price = parse_price(sales['prix_unitaire'])
print('Valeurs encore non convertibles :', unit_price.isna().sum())
print(unit_price.describe().round(2))

Valeurs encore non convertibles : 0
count    17864.00
mean       477.57
std        271.83
min         41.79
25%        217.46
50%        465.20
75%        707.19
max       1088.55
Name: prix_unitaire, dtype: float64


### Partie autonome

In [6]:
# Q1. Charger l'extrait des ventes depuis les trois formats disponibles,
#     puis comparer les dtypes obtenus pour la colonne 'date_commande'.

sample_csv = pd.read_excel(RAW_DIR / 'ventes_extrait.xlsx')  # TODO : attention, ventes_extrait.xlsx n'est pas un CSV
sample_xlsx = pd.read_json(RAW_DIR / 'ventes_extrait.json') # TODO : pd.read_excel
sample_json = pd.read_csv(RAW_DIR / 'ventes_brutes.csv', nrows=len(sample_xlsx))  # TODO : pd.read_json

for label, frame in [('csv', sample_csv), ('xlsx', sample_xlsx), ('json', sample_json)]:
    print(f"{label:<6} lignes={len(frame):<6} dtype date={frame['date_commande'].dtype}")

csv    lignes=1000   dtype date=str
xlsx   lignes=1000   dtype date=str
json   lignes=1000   dtype date=str


**Question à traiter par écrit dans la cellule suivante.** Les trois formats donnent-ils
le même type pour `date_commande` ? Le même nombre de lignes ? Que se passerait-il si votre
pipeline acceptait indifféremment ces trois sources ?

*Votre réponse :*
un pipeline qui accepte ces trois sources sans imposer les types produit des résultats incohérents selon l'origine du fichier. Il faut fixer le schéma à l'entrée, par exemple avec pd.to_datetime et dtype=.


In [7]:
# Q2. Relire ventes_brutes.csv en une seule instruction, en imposant :
#     - id_client, id_produit, id_magasin comme chaînes de caractères ;
#     - la chaîne vide et 'NC' comme valeurs manquantes ;
#     - seulement les colonnes id_commande, date_commande, id_produit, quantite,
#       prix_unitaire, statut.
#     Indice : paramètres dtype, na_values et usecols.

EXPECTED_COLUMNS = ['id_commande', 'date_commande', 'id_produit',
                    'quantite', 'prix_unitaire', 'statut']

sales_subset = pd.read_csv(
    RAW_DIR / 'ventes_brutes.csv',
    usecols=EXPECTED_COLUMNS,
    dtype={'id_client': object, 'id_produit': object, 'id_magasin': object},
    na_values=['', 'NC'],
)[EXPECTED_COLUMNS]  # TODO

assert list(sales_subset.columns) == EXPECTED_COLUMNS
assert sales_subset['id_produit'].dtype == object
print('OK —', sales_subset.shape)

OK — (17864, 6)


---
## Atelier 2.2 — Sélection et filtrage (60 min)

### Partie guidée : `loc`, `iloc` et le piège de la copie

In [8]:
sales['prix_unitaire'] = parse_price(sales['prix_unitaire'])
sales['montant'] = sales['quantite'] * sales['prix_unitaire']

# loc travaille sur les ÉTIQUETTES (noms de colonnes, valeurs d'index)
print(sales.loc[0:2, ['id_commande', 'quantite', 'montant']])
print()
# iloc travaille sur les POSITIONS entières
print(sales.iloc[0:2, [0, 5, -1]])

  id_commande  quantite  montant
0   CMD000001        11  9129.34
1   CMD000002         2   976.68
2   CMD000003         2  1476.60

  id_commande  quantite  montant
0   CMD000001        11  9129.34
1   CMD000002         2   976.68


Notez la différence sur les bornes : `loc[0:2]` renvoie **trois** lignes (borne incluse),
`iloc[0:2]` en renvoie **deux** (borne exclue, comme le slicing Python).

In [9]:
# Filtrage booléen : combiner des conditions avec & et |, chaque condition entre parenthèses
large_orders = sales[(sales['montant'] > 1000) & (sales['statut'] == 'livre')]
print('Commandes livrées de plus de 1000 EUR :', len(large_orders))

# query() est souvent plus lisible quand les conditions s'accumulent
same_result = sales.query("montant > 1000 and statut == 'livre'")
print('Même résultat :', len(same_result) == len(large_orders))

Commandes livrées de plus de 1000 EUR : 7387
Même résultat : True


In [10]:
# LE PIÈGE : modifier un sous-ensemble extrait par filtrage
cancelled = sales[sales['statut'] == 'annule']

# La ligne suivante déclenche un SettingWithCopyWarning : pandas ne sait pas si
# `cancelled` est une vue sur `sales` ou une copie indépendante.
cancelled['montant'] = 0

print('Montant dans sales pour les annulées :',
      sales.loc[sales['statut'] == 'annule', 'montant'].head(3).tolist())
print("-> la modification n'a PAS été propagée : le travail est perdu")

Montant dans sales pour les annulées : [9129.34, 1476.6, 6157.9800000000005]
-> la modification n'a PAS été propagée : le travail est perdu


**Les deux écritures correctes**, selon l'intention :

```python
# Intention A : modifier le DataFrame d'origine
sales.loc[sales['statut'] == 'annule', 'montant'] = 0

# Intention B : travailler sur une copie indépendante
cancelled = sales[sales['statut'] == 'annule'].copy()
cancelled['montant'] = 0
```

Le message d'avertissement est le symptôme d'une ambiguïté dans votre code, pas un bruit
à faire taire.

### Partie autonome

In [11]:
# Q3. Extraire les commandes qui remplissent TOUTES ces conditions :
#     - statut 'livre' ;
#     - quantite comprise entre 1 et 10 inclus ;
#     - montant strictement supérieur à la médiane des montants des commandes livrées ;
#     - canal contenant 'web', quelle que soit la casse.
#     Le résultat doit être une COPIE indépendante.

is_delivered = sales['statut'] == 'livre'
median_delivered = sales.loc[is_delivered, 'montant'].median()
channel = sales['canal'].str.strip().str.lower()

target_orders = sales[
    is_delivered
    & sales['quantite'].between(1, 10)
    & (sales['montant'] > median_delivered)
    & channel.str.contains('web', na=False)
].copy()  # TODO

assert isinstance(target_orders, pd.DataFrame)
assert target_orders['quantite'].between(1, 10).all()
assert (target_orders['statut'] == 'livre').all()
assert target_orders['canal'].str.strip().str.lower().eq('web').all()
print('OK —', len(target_orders), 'commandes retenues')

OK — 1446 commandes retenues


In [12]:
# Q4. Sans utiliser groupby, calculer le montant total des commandes livrées
#     pour chacun des trois canaux (après normalisation de la casse).
#     Un dictionnaire {canal: total} est attendu.
channel = sales['canal'].str.strip().str.lower()
is_delivered = sales['statut'] == 'livre'

revenue_by_channel = {
    c: sales.loc[is_delivered & (channel == c), 'montant'].sum()
    for c in channel.dropna().unique()
}  # TODO

assert set(revenue_by_channel) == {'web', 'boutique', 'telephone'}
print('OK')
for channel, total in sorted(revenue_by_channel.items(), key=lambda item: -item[1]):
    print(f'  {channel:<12} {total:>14,.2f} EUR'.replace(',', ' '))

OK
  boutique       9 909 902.59 EUR
  web            9 849 162.40 EUR
  telephone      9 750 831.44 EUR


---
## Atelier 2.3 — Un rapport d'exploration réutilisable (100 min)

Face à un jeu de données inconnu, les mêmes questions reviennent toujours. Plutôt que de
les reposer à la main à chaque fois, vous allez écrire une fonction qui y répond.
**Cette fonction vous servira jusqu'à la fin du module, y compris sur votre projet.**

### Partie guidée : les briques

In [13]:
print('--- shape ---'); print(sales.shape)
print('\n--- info ---'); sales.info(memory_usage='deep')

--- shape ---
(17864, 10)

--- info ---
<class 'pandas.DataFrame'>
RangeIndex: 17864 entries, 0 to 17863
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id_commande    17864 non-null  str    
 1   date_commande  17864 non-null  str    
 2   id_client      17684 non-null  str    
 3   id_magasin     17703 non-null  str    
 4   id_produit     17713 non-null  str    
 5   quantite       17864 non-null  int64  
 6   prix_unitaire  17864 non-null  float64
 7   statut         17864 non-null  str    
 8   canal          17864 non-null  str    
 9   montant        17864 non-null  float64
dtypes: float64(2), int64(1), str(7)
memory usage: 2.2 MB


In [14]:
print('--- taux de valeurs manquantes ---')
missing_rate = (sales.isna().mean() * 100).round(2).sort_values(ascending=False)
print(missing_rate[missing_rate > 0])

print('\n--- cardinalité des colonnes texte ---')
for column in sales.select_dtypes(include='object').columns:
    print(f'  {column:<18} {sales[column].nunique():>6} modalités')

--- taux de valeurs manquantes ---
id_client     1.01
id_magasin    0.90
id_produit    0.85
dtype: float64

--- cardinalité des colonnes texte ---
  id_commande         17600 modalités
  date_commande        7734 modalités
  id_client            2993 modalités
  id_magasin             10 modalités
  id_produit             40 modalités
  statut                  4 modalités
  canal                   9 modalités


In [15]:
print('--- modalités de canal, telles quelles ---')
print(sales['canal'].value_counts())

--- modalités de canal, telles quelles ---
canal
boutique        4787
telephone       4764
web             4724
WEB              702
BOUTIQUE         701
TELEPHONE        665
  web            551
  telephone      487
  boutique       483
Name: count, dtype: int64


Neuf modalités pour ce qui devrait en compter trois. La casse et les espaces parasites
créent de faux niveaux. Un `value_counts()` brut est précisément l'outil qui révèle ce
genre de problème : c'est pourquoi il doit figurer dans le rapport automatique.

### Partie autonome : la fonction `profile_dataframe`

In [21]:
def profile_dataframe(df, name='jeu de données', max_cardinality=25):
"""Affiche un rapport d'exploration standard.
 
Doit produire, dans cet ordre :
1. le nom, les dimensions et l'empreinte mémoire ;
2. le nombre de lignes strictement dupliquées ;
3. un tableau par colonne : type, nombre de valeurs manquantes,
taux en %, nombre de valeurs distinctes ;
4. les statistiques descriptives des colonnes numériques ;
5. pour chaque colonne texte de cardinalité inférieure à
max_cardinality, la répartition des modalités.
 
Ne renvoie rien : la fonction affiche.
"""
 
memory_mb = df.memory_usage(deep=True).sum() / 1024**2
 
print(f'=== {name} ===')
print(
f'Dimensions : {df.shape[0]} lignes x {df.shape[1]} colonnes'
f' | mémoire : {memory_mb:.2f} Mo'
)
 
print(f'Lignes dupliquées : {df.duplicated().sum()}')
 
summary = pd.DataFrame({
'type': df.dtypes.astype(str),
'manquants': df.isna().sum(),
'taux_%': (df.isna().mean() * 100).round(2),
'distinctes': df.nunique()
})
 
print('\n--- colonnes ---')
print(summary.to_string())
 
numeric = df.select_dtypes(include='number')
 
if not numeric.empty:
print('\n--- statistiques numériques ---')
print(numeric.describe().T.round(2).to_string())
 
for column in df.select_dtypes(include=['object', 'string']).columns:
n_unique = df[column].nunique()
 
if n_unique < max_cardinality:
print(f'\n--- {column} ({n_unique} modalités) ---')
print(df[column].value_counts(dropna=False).to_string())
print()
profile_dataframe(sales, name='ventes_brutes')

IndentationError: expected an indented block after function definition on line 1 (2582697105.py, line 2)

In [ ]:
# Q5. Appliquer la fonction aux deux autres jeux de données et vérifier qu'elle
#     se comporte correctement sur des structures différentes.

customers = pd.read_csv(RAW_DIR / 'clients.csv')
sensors = pd.read_csv(RAW_DIR / 'capteurs.csv')

profile_dataframe(customers, name='clients')
profile_dataframe(sensors, name='capteurs')

In [ ]:
# Q6. Déplacer la fonction dans src/exploration.py, puis l'importer ici.
#     Le notebook doit rester lisible : le code réutilisable vit dans src/.

import sys
sys.path.insert(0, str(ROOT / 'src'))

# from exploration import profile_dataframe   # décommentez une fois le fichier créé

---
## Ce que le rapport révèle déjà

Rédigez ici, en cinq à dix lignes, la liste des anomalies que votre rapport a mises au jour
sur `ventes_brutes`. Ce texte est le point de départ de la séance 3 et le premier élément
de votre note méthodologique de projet.

*Vos observations :*

1. 
2. 
3. 

---
## Livrable de la séance

- `src/exploration.py` contenant la fonction `profile_dataframe`, importable ;
- ce notebook exécuté, avec les six questions complétées ;
- la liste écrite des anomalies constatées.